In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load raw dataset ONCE from cache/network and save offline fallback
df_raw = sns.load_dataset('titanic')
df_raw.to_csv("titanic.csv", index=False)
print(f"Dataset shape: {df_raw.shape}")

# Reload from local offline fallback
df = pd.read_csv("titanic.csv")

# 2. Missing values percentage
missing_pct = (df.isnull().sum() / len(df)) * 100
print("Missing percentage per column:\n", missing_pct[missing_pct > 0])

# Strategy application:
# - emb_town / embarked: < 5% missing -> drop rows
# - age: 19.87% missing (5-30%) -> impute with median
# - deck: 77.10% missing (>30%) -> drop column or encode 'missing' as distinct category (dropping column)

df_clean = df.drop(columns=['deck']).copy()
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
df_clean = df_clean.dropna(subset=['embarked', 'embark_town'])

# 3. Outlier check & skewness analysis
for col in ['age', 'fare']:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    outliers = df_clean[(df_clean[col] < (q1 - 1.5 * iqr)) | (df_clean[col] > (q3 + 1.5 * iqr))]
    print(f"Outliers in {col}: {len(outliers)}")

fare_mean = df_clean['fare'].mean()
fare_median = df_clean['fare'].median()
fare_mode = df_clean['fare'].mode()[0]
print(f"Fare Mean: {fare_mean:.2f}, Median: {fare_median:.2f}, Mode: {fare_mode:.2f}")

# 4. Bivariate Analysis
print("\nSurvival by Sex:\n", df_clean.groupby('sex')['survived'].mean())
print("\nSurvival by Class:\n", df_clean.groupby('pclass')['survived'].mean())
print("\nSurvival by Sex & Class:\n", df_clean.groupby(['sex', 'pclass'])['survived'].mean())

# Correlation heatmap on specified 6 columns ONLY (excluding adult_male, alone)
corr_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
corr_matrix = df_clean[corr_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix (6 Core Numeric Attributes)")
plt.savefig("correlation_heatmap.png")
plt.close()

# 5. Exploratory Standardization check
from sklearn.preprocessing import StandardScaler
scaler_eda = StandardScaler()
scaled_vals = scaler_eda.fit_transform(df_clean[['age', 'fare']])
print(f"Standardized Mean: {scaled_vals.mean(axis=0).round(4)}")
print(f"Standardized Std: {scaled_vals.std(axis=0).round(4)}")